In [83]:
import numpy as np
from tqdm import tqdm, trange
from sklearn.linear_model import Lasso
from sklearn.ensemble import RandomForestRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_squared_error
from joblib import Parallel, delayed
from xgboost import XGBRegressor
from scipy.linalg import cholesky, cho_solve
import multiprocessing

In [151]:
# Settings
MC = 25
penalty = 0.1
q_list = [6, 4, 2, 0]  # decays to test
r = None  # will set to min(p, q)
tol, T = 5e-3, 1000
n_theta = 5

# Load and normalize data
Z = np.load("X_news_embeddings_unique.npy")
X = np.load("X_user_features_unique.npy")
Z = Z / np.linalg.norm(Z, axis=1, keepdims=True)
X = X / np.linalg.norm(X, axis=1, keepdims=True)

p, q = Z.shape[1], X.shape[1]
rrr = min(p, q)
N = 5000
np.random.seed(2025)

In [108]:
def DKRL_dual(Z, X, y, r, penalty, tol, T):
    p = Z.shape[1]
    q = X.shape[1]
    N = Z.shape[0]
    
    Ut = np.random.normal(loc=0, scale=1, size=(p, r))
    Vt = np.random.normal(loc=0, scale=1, size=(q, r))
    Utt = Ut.copy()
    Vtt = Vt.copy()

    for iter in tqdm(range(T)):
        # update U matrix    
        hhat = np.dot(X, Vtt)
        for i in range(r):
            # Compute residuals first
            ghat = np.dot(Z, Utt)
            product = ghat * hhat
            product[:, i] = 0
            y_pred = product.sum(axis=1)
            y_residual = y - y_pred
            DZ = Z * hhat[:, i][:, np.newaxis]

            # Update Utt[:, i]
            Utt[:, i] = np.linalg.solve(np.dot(DZ.T, DZ) + penalty * np.eye(p), np.dot(DZ.T, y_residual))
            

        # update V matrix
        ghat = np.dot(Z, Utt)
        for i in range(r):
            # Compute residuals first
            hhat = np.dot(X, Vtt)
            product = ghat * hhat
            product[:, i] = 0
            y_pred = product.sum(axis=1)
            y_residual = y - y_pred
            DX = X * ghat[:, i][:, np.newaxis]
            
            # Update Vtt[:, i]
            Vtt[:, i] = np.linalg.solve(np.dot(DX.T, DX) + penalty * np.eye(q), np.dot(DX.T, y_residual))

        # check stopping criterion
        # print(iter)
        # print(np.linalg.norm(Utt - Ut)/(np.linalg.norm(Ut) + 1e-4))
        if (np.linalg.norm(Utt - Ut)/(np.linalg.norm(Ut) + 1e-4) < tol and np.linalg.norm(Vtt - Vt)/(np.linalg.norm(Vt) + 1e-3) < tol):
            break

        # if not stop, update Ut and Vt
        Ut = Utt.copy()
        Vt = Vtt.copy()

    y_pred = (np.dot(Z, Utt) * np.dot(X, Vtt)).sum(axis=1)
    return Utt, Vtt, y_pred

def DKRL_pred_dual(U, V, Z, X):
    return ((Z @ U) * (X @ V)).sum(axis=1)

In [133]:
import numpy as np
from tqdm import tqdm

def DKRL_dual_sor(Z, X, y, r, penalty, tol, T, omega=1.1):
    """
    DKRL_dual with per-coordinate SOR over-relaxation.

    Args:
      Z:        (N×p) feature matrix for Z-side
      X:        (N×q) feature matrix for X-side
      y:        (N,)  response vector
      r:        rank
      penalty: regularization weight (scalar)
      tol:     convergence tolerance
      T:       max # outer iterations
      omega:   relaxation factor (>0; ω=1 is no relaxation)

    Returns:
      Utt:   (p×r) learned U
      Vtt:   (q×r) learned V
      y_pred (N,) predictions
    """
    N, p = Z.shape
    q    = X.shape[1]

    # Initialize factors
    Utt = np.random.randn(p, r)
    Vtt = np.random.randn(q, r)
    Ut_prev, Vt_prev = Utt.copy(), Vtt.copy()

    for outer in tqdm(range(T)):
        # --- U-sweep with SOR ---
        hhat = X @ Vtt                # (N x r)
        for i in range(r):
            # residual leaving out component i
            ghat       = Z @ Utt      # (N x r)
            prod       = ghat * hhat
            prod[:, i] = 0
            y_residual = y - prod.sum(axis=1)

            # normal equations for coordinate i
            DZ    = Z * hhat[:, i:i+1]           # (N x p)
            A     = DZ.T @ DZ + penalty*np.eye(p)
            b     = DZ.T @ y_residual
            u_raw = np.linalg.solve(A, b)        # raw update

            # SOR relaxation step
            delta_u      = u_raw - Utt[:, i]
            Utt[:, i]   += omega * delta_u

        # --- V-sweep with SOR ---
        ghat = Z @ Utt                # (N x r)
        for i in range(r):
            # residual leaving out component i
            hhat       = X @ Vtt      # (N x r)
            prod       = ghat * hhat
            prod[:, i] = 0
            y_residual = y - prod.sum(axis=1)

            # normal equations for coordinate i
            DX    = X * ghat[:, i:i+1]          # (N x q)
            A     = DX.T @ DX + penalty*np.eye(q)
            b     = DX.T @ y_residual
            v_raw = np.linalg.solve(A, b)       # raw update

            # SOR relaxation
            delta_v      = v_raw - Vtt[:, i]
            Vtt[:, i]   += omega * delta_v

        # --- check convergence on U and V ---
        du = np.linalg.norm(Utt - Ut_prev) / (np.linalg.norm(Ut_prev) + 1e-4)
        dv = np.linalg.norm(Vtt - Vt_prev) / (np.linalg.norm(Vt_prev) + 1e-4)
        if du < tol and dv < tol:
            break

        # update history
        Ut_prev, Vt_prev = Utt.copy(), Vtt.copy()

    # final preds
    y_pred = ((Z @ Utt) * (X @ Vtt)).sum(axis=1)
    return Utt, Vtt, y_pred



In [152]:
def run_simulation(Z, X, Theta, penalty, N, r, tol, T):
    # Sample with replacement
    idx_Z = np.random.choice(Z.shape[0], N, replace=True)
    idx_X = np.random.choice(X.shape[0], N, replace=True)
    Zs, Xs = Z[idx_Z], X[idx_X]
    y = np.diag(Zs @ Theta @ Xs.T)

    # Train-test split
    N_train = int(0.9 * N)
    train_idx = np.random.choice(N, N_train, replace=False)
    test_idx = np.setdiff1d(np.arange(N), train_idx)
    Z_tr, X_tr, y_tr = Zs[train_idx], Xs[train_idx], y[train_idx]
    Z_te, X_te, y_te = Zs[test_idx], Xs[test_idx], y[test_idx]

    # Kernel matrices
    KZ_tr = Z_tr @ Z_tr.T
    KX_tr = X_tr @ X_tr.T
    KZX_tr = KZ_tr * KX_tr
    KZ_te = Z_tr @ Z_te.T
    KX_te = X_tr @ X_te.T
    KZX_te = KZ_te * KX_te

    out = {}
    # DKRL
    U, V, y_hat = DKRL_dual_sor(Z_tr, X_tr, y_tr, r, penalty, tol, T)
    y_pred = DKRL_pred_dual(U, V, Z_te, X_te)
    out['DKRL'] = [np.mean((y_tr - y_hat)**2), np.mean((y_te - y_pred)**2)]

    # Only Z
    # Lasso Z
    lz = Lasso(alpha=1e-4, max_iter=1000)
    out['Lasso_Z'] = [np.mean((y_tr - lz.fit(Z_tr, y_tr).predict(Z_tr))**2), np.mean((y_te - lz.predict(Z_te))**2)]
    # XGB Z
    xz = XGBRegressor(n_estimators=100, n_jobs=1, eval_metric='rmse')
    out['XGB_Z'] = [np.mean((y_tr - xz.fit(Z_tr, y_tr).predict(Z_tr))**2), np.mean((y_te - xz.predict(Z_te))**2)]
    # FNN Z
    fz = MLPRegressor(hidden_layer_sizes=(100,), max_iter=500)
    out['FNN_Z'] = [np.mean((y_tr - fz.fit(Z_tr, y_tr).predict(Z_tr))**2), np.mean((y_te - fz.predict(Z_te))**2)]
    # Kernel Z
    try:
        az = np.linalg.solve(KZ_tr + penalty * np.eye(N_train), y_tr)
        yhz = KZ_tr @ az; ypz = KZ_te.T @ az
        out['Kernel_Z'] = [np.mean((y_tr - yhz)**2), np.mean((y_te - ypz)**2)]
    except np.linalg.LinAlgError:
        out['Kernel_Z'] = [np.nan, np.nan]

    # Only X
    # Lasso X
    lx = Lasso(alpha=1e-4, max_iter=1000)
    out['Lasso_X'] = [np.mean((y_tr - lx.fit(X_tr, y_tr).predict(X_tr))**2), np.mean((y_te - lx.predict(X_te))**2)]
    # XGB X
    xx = XGBRegressor(n_estimators=100, n_jobs=1, eval_metric='rmse')
    out['XGB_X'] = [np.mean((y_tr - xx.fit(X_tr, y_tr).predict(X_tr))**2), np.mean((y_te - xx.predict(X_te))**2)]
    # FNN X
    fx = MLPRegressor(hidden_layer_sizes=(100,), max_iter=500)
    out['FNN_X'] = [np.mean((y_tr - fx.fit(X_tr, y_tr).predict(X_tr))**2), np.mean((y_te - fx.predict(X_te))**2)]
    # Kernel X
    try:
        ax = np.linalg.solve(KX_tr + penalty * np.eye(N_train), y_tr)
        yhx = KX_tr @ ax; ypx = KX_te.T @ ax
        out['Kernel_X'] = [np.mean((y_tr - yhx)**2), np.mean((y_te - ypx)**2)]
    except np.linalg.LinAlgError:
        out['Kernel_X'] = [np.nan, np.nan]

    # ZX methods
    XZX_tr = np.hstack([Z_tr, X_tr]); XZX_te = np.hstack([Z_te, X_te])
    # Lasso ZX
    lzx = Lasso(alpha=1e-4, max_iter=1000)
    out['Lasso_ZX'] = [np.mean((y_tr - lzx.fit(XZX_tr, y_tr).predict(XZX_tr))**2), np.mean((y_te - lzx.predict(XZX_te))**2)]
    # XGB ZX
    xzx = XGBRegressor(n_estimators=100, n_jobs=1, eval_metric='rmse')
    out['XGB_ZX'] = [np.mean((y_tr - xzx.fit(XZX_tr, y_tr).predict(XZX_tr))**2), np.mean((y_te - xzx.predict(XZX_te))**2)]
    # FNN ZX
    fzx = MLPRegressor(hidden_layer_sizes=(100,), max_iter=500)
    out['FNN_ZX'] = [np.mean((y_tr - fzx.fit(XZX_tr, y_tr).predict(XZX_tr))**2), np.mean((y_te - fzx.predict(XZX_te))**2)]
    # Kernel ZX
    try:
        axz = np.linalg.solve(KZX_tr + 5e1 * np.eye(N_train), y_tr)
        yhzx = KZX_tr @ axz; ypzx = KZX_te.T @ axz
        out['Kernel_ZX'] = [np.mean((y_tr - yhzx)**2), np.mean((y_te - ypzx)**2)]
    except np.linalg.LinAlgError:
        out['Kernel_ZX'] = [np.nan, np.nan]

    return out

# Main loop
results = {q: {'DKRL': [],
               'Lasso_Z': [], 'XGB_Z': [], 'FNN_Z': [], 'Kernel_Z': [],
               'Lasso_X': [], 'XGB_X': [], 'FNN_X': [], 'Kernel_X': [],
               'Lasso_ZX': [], 'XGB_ZX': [], 'FNN_ZX': [], 'Kernel_ZX': []}
           for q in q_list}
for q_decay in q_list:
    eigs = np.concatenate([np.ones(5), np.array([i**(-q_decay) for i in range(6, rrr+1)])])
    Sigma = np.diag(eigs)
    for _ in trange(n_theta, desc=f"Theta loops q={q_decay}"):
        P = np.random.normal(0,1,(p,q))
        L, _, Rt = np.linalg.svd(P, full_matrices=False)
        Lr, Rr = L[:,:rrr], Rt[:rrr,:].T
        Theta = Lr @ Sigma @ Rr.T
        outputs = Parallel(n_jobs=10)(
            delayed(run_simulation)(Z, X, Theta, penalty = 1e-1, N = N, r = 10, tol = 5e-3, T = 1000)
            for _ in trange(MC, desc="MC sims", leave=False)
        )
        for res in outputs:
            for method, errs in res.items():
                results[q_decay][method].append(errs)

# Save results
np.save("MC_simulation_results_fixed_penalty_full.npy", results)


 21%|██        | 206/1000 [00:16<00:47, 16.57it/s]/Users/leis/Documents/GitHub/DKRL-LLM/dkrl-py312/lib/python3.12/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(
Theta loops q=0: 100%|██████████| 5/5 [05:28<00:00, 65.70s/it]


In [150]:
results

{6: {'DKRL': [[np.float64(1.6456869079782152e-05),
    np.float64(2.3655864575882735e-05)],
   [np.float64(3.550965589809132e-06), np.float64(4.239486603013149e-06)],
   [np.float64(6.691844840580788e-06), np.float64(6.950641250113854e-06)],
   [np.float64(5.777637660264648e-06), np.float64(7.028089875643115e-06)],
   [np.float64(1.1281666047973642e-05), np.float64(1.5294549725148397e-05)]],
  'Lasso_Z': [[np.float64(7.164477423083837e-05),
    np.float64(7.955938182580165e-05)],
   [np.float64(8.010509993326826e-05), np.float64(9.975701936600145e-05)],
   [np.float64(8.200607251546855e-05), np.float64(8.612431410978571e-05)],
   [np.float64(5.091357270480492e-05), np.float64(5.6140498351958666e-05)],
   [np.float64(0.00010657451057552672), np.float64(0.00011094446279799967)]],
  'XGB_Z': [[np.float64(2.6617278460462224e-05),
    np.float64(4.686707028515063e-05)],
   [np.float64(3.45976611317791e-05), np.float64(7.104917895079627e-05)],
   [np.float64(3.525875607070075e-05), np.float6

In [ ]:
# import the headlines and embeddings
num_actions = 100

# Load and normalize data
Z = np.load("X_news_embeddings_unique.npy")
X = np.load("X_user_features_unique.npy")
Z = Z / np.linalg.norm(Z, axis=1, keepdims=True)
X = X / np.linalg.norm(X, axis=1, keepdims=True)

p, q = Z.shape[1], X.shape[1]
 

# Randomly sample a subset of headlines and embeddings for evaluation
# Set a random seed for reproducibility
np.random.seed(2025)
ind = np.arange(len(headlines))  # Array with 1000 elements
sampled_ind = np.random.choice(ind, size=num_actions, replace=False)

# Randomly sample 100 elements from the array
Z = embeddings[sampled_ind]

# dimension of the embeddings
p = len(Z[1])
print("the dimension of the embeddings is: " + str(len(Z[1])))

# ============================================================================ #

# Simulate projection matrix
# Simulate covariate level features from Gaussian distributions
np.random.seed(2025)
r = 2
# p = 1000
# Z = np.random.normal(loc=0, scale=1, size=(N, p))
# row_norms_Z = np.linalg.norm(Z, axis=1, keepdims=True)
# Z = Z / row_norms_Z
# sampled_ind = np.random.choice(range(N), size=N, replace=True)
# Z = Z[sampled_ind]

q = 1000
X = np.random.normal(loc=0, scale=1, size=(num_actions, q))
row_norms_X = np.linalg.norm(X, axis=1, keepdims=True)
X = X / row_norms_X

# generate random projection
np.random.seed(2025)
P = np.random.normal(loc=0, scale=1, size=(p, q))
L, S, Rt = np.linalg.svd(P, full_matrices=False)
Lr = L[:, :r]
Rr = Rt[:r, :].T
Zr = np.dot(Z, Lr) # n*r
Xr = np.dot(X, Rr) # n*r

# True reward matrix
true_reward = Zr @ Xr.T
best_reward = np.max(true_reward, axis=0)

# ============================================================================ #

# Set the random seed for reproducibility
np.random.seed(2025)

# Define the number of simulations
num_simulations = 100
num_exploration = 1000
num_exploitation = 3000
num_actions = 100  # Assuming actions correspond to columns

# Store cumulative regrets
all_cum_regrets = np.zeros((num_simulations, num_exploration + num_exploitation))

for sim in tqdm(range(num_simulations)):
    # ========= Exploration Stage ==========
    sampled_ind_z = np.random.choice(range(num_actions), size=num_exploration, replace=True)
    Zsub = Z[sampled_ind_z]
    sampled_ind_x = np.random.choice(range(num_actions), size=num_exploration, replace=True)
    Xsub = X[sampled_ind_x]

    Zrsub = np.dot(Zsub, Lr)  # n*r
    Xrsub = np.dot(Xsub, Rr)  # n*r
    KZsub = np.dot(Zsub, Zsub.T)
    KXsub = np.dot(Xsub, Xsub.T)

    KZ_test = np.dot(Zsub, Z.T)
    KX_test = np.dot(Xsub, X.T)

    observed_reward_exploration = true_reward[sampled_ind_z, sampled_ind_x]
    best_reward_exploration = best_reward[sampled_ind_x]

    # Generate the outcome using DKRL
    y = np.diag(Zrsub @ Xrsub.T)
    U, V, y_hat = DKRL(y, KZsub, KXsub, num_exploration, r, penalty, tol, T, wt=0.01)
    y_pred = DKRL_pred(U, V, KZ_test, KX_test)
    reward_pred = KZ_test.T @ U @ V.T @ KX_test

    row_indices = np.argmax(reward_pred, axis=0)
    best_reward_hat = true_reward[row_indices, range(num_actions)]

    # ========= Exploitation Stage ==========
    sampled_ind_xx = np.random.choice(range(num_actions), size=num_exploitation, replace=True)
    observed_reward_exploitation = best_reward_hat[sampled_ind_xx]
    best_reward_exploitation = best_reward[sampled_ind_xx]

    # Compute regret
    regret = np.append(
        best_reward_exploration - observed_reward_exploration,
        best_reward_exploitation - observed_reward_exploitation
    )

    # Compute cumulative regret
    cum_regret = np.cumsum(regret)
    
    # Store results
    all_cum_regrets[sim, :] = cum_regret



np.save("MC_simulation_results_regret.npy", all_cum_regrets)